In [1]:
!pip install torch numpy scipy wfdb scikit-learn matplotlib --quiet

In [2]:
# ==============================================================================
# Kaggle Parameter Tuning Script (Stage 1: Network Optimization)
# Powered by Optuna
# ==============================================================================
import os
import time
import sys
from collections import defaultdict

# 1. Define the path to your dataset containing the .py files
# Make sure to replace 'your-dataset-name' with the actual folder name in Kaggle
MODULE_DIR = '/kaggle/input/datasets/oshanimaduwage/code-files'

# 2. Add the directory to Python's system path so it can find the modules
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)

import torch
import torch.nn as nn
import optuna
from sklearn.metrics import f1_score

# --- Import from your Kaggle Utility Scripts ---
from PPG_Breath_Model_Preprocessing import Config, BIDMCLoader, subject_split
from PPG_Breath_Model_Dataset import RespiratoryWindowDataset, make_loader
from PPG_Breath_Model_Architecture import RespiratoryDetector

cfg = Config()

# Replace with the folder name of your uploaded BIDMC dataset
cfg.BIDMC_DIR = "/kaggle/input/datasets/oshanimaduwage/bidmc-ppg-and-respiration-dataset-1-0-0/bidmc-ppg-and-respiration-dataset-1.0.0"

# Keep your tuning parameters consistent
cfg.WINDOW_S = 3.0
cfg.STRIDE_S = 0.30
cfg.BATCH_SIZE = 64

In [ ]:
# ==============================================================================
# 1. FIXED ARCHITECTURE & LOSS (From Previous Diagnosis)
# ==============================================================================
class RobustRegressionHead(nn.Module):
    """Replaces BatchNorm with LayerNorm to prevent empty-window washouts."""
    def __init__(self, in_features, num_transitions, window_samples, dropout):
        super().__init__()
        self.window_samples = float(window_samples)
        self.net = nn.Sequential(
            nn.Linear(in_features, in_features // 2),
            nn.LayerNorm(in_features // 2), 
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(in_features // 2, num_transitions),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x) * self.window_samples

# Monkey-patch the detector
original_init = RespiratoryDetector.__init__
def patched_init(self, cfg):
    original_init(self, cfg)
    self.reg_fusion = RobustRegressionHead(
        cfg.DENSE_UNITS, cfg.NUM_TRANSITIONS, cfg.WINDOW_SAMPLES, cfg.DROPOUT)
    self.reg_primary = RobustRegressionHead(
        cfg.DENSE_UNITS, cfg.NUM_TRANSITIONS, cfg.WINDOW_SAMPLES, cfg.DROPOUT)
RespiratoryDetector.__init__ = patched_init

class MaskedCompositeLoss(nn.Module):
    """Uses SmoothL1Loss (Huber) for stable regression gradients."""
    def __init__(self, cfg):
        super().__init__()
        self.alpha = cfg.ALPHA
        self.window_samples = float(cfg.WINDOW_SAMPLES)
        self.register_buffer("pos_weight", torch.full((cfg.NUM_TRANSITIONS,), cfg.POS_WEIGHT))

    def forward(self, preds, targets):
        t_prob, t_loc = targets["t_prob"].float(), targets["t_loc"].float()
        
        # Classification
        bce = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(t_prob.device))
        l_cls = (bce(preds["logits_fusion"], t_prob) + bce(preds["logits_primary"], t_prob)) / 2.0

        # Regression
        mask = (t_loc >= 0.0).float()
        loc_gt_safe = t_loc.clamp(min=0.0)
        n_valid = mask.sum().clamp(min=1.0)

        loc_fusion_norm = preds["loc"] / self.window_samples
        loc_gt_norm     = loc_gt_safe / self.window_samples
        l1_fusion = nn.functional.smooth_l1_loss(loc_fusion_norm, loc_gt_norm, reduction='none')
        l_reg_fusion = (l1_fusion * mask).sum() / n_valid

        if preds["loc_primary"] is not None:
            loc_primary_norm = preds["loc_primary"] / self.window_samples
            l1_primary = nn.functional.smooth_l1_loss(loc_primary_norm, loc_gt_norm, reduction='none')
            l_reg_primary = (l1_primary * mask).sum() / n_valid
            # l_reg = (l_reg_fusion + l_reg_primary) / 2.0
            # Force the regression loss to be on the same magnitude scale as BCE
            l_reg = ((l_reg_fusion + l_reg_primary) / 2.0) * 10.0
        else:
            l_reg = l_reg_fusion * 10.0

        return {"total": self.alpha * l_cls + (1.0 - self.alpha) * l_reg, "cls": l_cls, "reg": l_reg}



In [ ]:
# ==============================================================================
# 2. GLOBAL SETUP & DATA LOADING (Run ONCE to save time)
# ==============================================================================

cfg = Config()
cfg.BIDMC_DIR = "/kaggle/input/datasets/oshanimaduwage/bidmc-ppg-and-respiration-dataset-1-0-0/bidmc-ppg-and-respiration-dataset-1.0.0"
cfg.WINDOW_S = 3.0    # Fixed for network tuning
cfg.STRIDE_S = 0.30
cfg.BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading dataset into RAM once...")
bidmc_loader = BIDMCLoader(cfg)
all_recordings = bidmc_loader.load_all()
train_recs, val_recs = subject_split(all_recordings, val_fraction=0.15, seed=42)

# We build datasets outside the tuning loop so we don't waste time recalculating them
train_ds = RespiratoryWindowDataset(train_recs, cfg, augment=True)
val_ds   = RespiratoryWindowDataset(val_recs,   cfg, augment=False)
train_loader = make_loader(train_ds, cfg, train=True,  num_workers=2)
val_loader   = make_loader(val_ds,   cfg, train=False, num_workers=2)

In [ ]:
# ==============================================================================
# 3. OPTUNA OBJECTIVE FUNCTION
# ==============================================================================
def objective(trial):
    """
    This function is executed by Optuna for every trial.
    It returns the metrics we want to optimize (F1 and MAE).
    """
    # --- A. Suggest Hyperparameters ---
    cfg.LR          = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    cfg.ALPHA       = trial.suggest_float("alpha", 0.3, 0.7)
    cfg.DROPOUT     = trial.suggest_float("dropout", 0.1, 0.5)
    cfg.POS_WEIGHT  = trial.suggest_float("pos_weight", 1.0, 4.0)
    
    # You can also tune architectural elements!
    cfg.DENSE_UNITS = trial.suggest_categorical("dense_units", [64, 128, 256])

    # --- B. Initialize Model for this Trial ---
    model = RespiratoryDetector(cfg).to(device)
    
    # Trigger lazy initialization
    dummy_input = torch.zeros(2, 1, cfg.WINDOW_SAMPLES, device=device)
    _ = model(dummy_input, dummy_input, dummy_input)

    loss_fn = MaskedCompositeLoss(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler()

    epochs = 15 # Keep epochs low (10-20) for rapid hyperparameter searching
    best_f1 = 0.0
    best_mae = float('inf')

    # --- C. Fast Training Loop ---
    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            p, a, m = batch["primary"].to(device), batch["auxiliary"].to(device), batch["aux_marker1"].to(device)
            t_prob, t_loc = batch["t_prob"].to(device), batch["t_loc"].to(device)

            with torch.autocast('cuda', enabled=True):
                preds = model(p, a, m)
                losses = loss_fn(preds, {"t_prob": t_prob, "t_loc": t_loc})

            scaler.scale(losses["total"]).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            scaler.step(optimizer)
            scaler.update()

        # --- D. Fast Evaluation Loop ---
        model.eval()
        total_mae_ms, valid_batches = 0.0, 0
        all_probs, all_labels = [], []

        with torch.no_grad():
            for batch in val_loader:
                p, a, m = batch["primary"].to(device), batch["auxiliary"].to(device), batch["aux_marker1"].to(device)
                t_prob, t_loc = batch["t_prob"].to(device), batch["t_loc"].to(device)
                
                with torch.autocast('cuda', enabled=True):
                    preds = model(p, a, m)
                
                # Regressor Metric
                mask = (t_loc >= 0.0)
                if mask.any():
                    err_samples = torch.abs(preds["loc"][mask] - t_loc[mask])
                    total_mae_ms += ((err_samples / cfg.TARGET_FS) * 1000.0).mean().item()
                    valid_batches += 1

                # Classifier Metric
                all_probs.append(preds["prob"].cpu())
                all_labels.append(t_prob.cpu())

        val_mae = total_mae_ms / valid_batches if valid_batches > 0 else 999.0
        
        probs = torch.cat(all_probs).numpy()
        labels = torch.cat(all_labels).numpy()
        preds_bin = (probs >= 0.5).astype(int)
        val_f1 = f1_score(labels[:, 0], preds_bin[:, 0], zero_division=0)

        # Track best metrics within this trial
        best_f1 = max(best_f1, val_f1)
        best_mae = min(best_mae, val_mae)

    # Optuna requires us to return the metrics we want to optimize
    return best_f1, best_mae

In [ ]:
# ==============================================================================
# 4. RUNNING THE STUDY
# ==============================================================================
if __name__ == "__main__":
    print("\nStarting Optuna Hyperparameter Sweep...")
    
    # We use a SQLite database. If Kaggle times out, you can re-run the notebook 
    # and it will pick up exactly where it left off!
    db_path = "/kaggle/working/tuning_study.db"
    
    # MULTI-OBJECTIVE: We want to Maximize F1 (direction 1) and Minimize MAE (direction 2)
    study = optuna.create_study(
        study_name="respiratory_tuning",
        storage=f"sqlite:///{db_path}",
        load_if_exists=True,
        directions=["maximize", "minimize"] 
    )

    # Run 30 trials (Adjust based on your Kaggle GPU time limit)
    study.optimize(objective, n_trials=30)

    print("\n--- Tuning Complete ---")
    print(f"Number of finished trials: {len(study.trials)}")
    
    # Because it's multi-objective, there is no single "best" trial. 
    # Optuna returns a "Pareto Front" (the best trade-offs between F1 and MAE).
    print("\nPareto Front (Best Trade-offs):")
    for trial in study.best_trials:
        print(f"  Trial {trial.number}: F1={trial.values[0]:.3f}, MAE={trial.values[1]:.1f}ms")
        print(f"    Params: {trial.params}")

In [3]:
# ==============================================================================
# MANUAL TEST: 5-Epoch Run with 10x Regression Loss Multiplier (FIXED)
# ==============================================================================
import time
from collections import defaultdict
import torch
import torch.nn as nn
from sklearn.metrics import f1_score

from PPG_Breath_Model_Preprocessing import Config, BIDMCLoader, subject_split
from PPG_Breath_Model_Dataset import RespiratoryWindowDataset, make_loader
from PPG_Breath_Model_Architecture import RespiratoryDetector

# 1. Configuration (Restricted to 5 Epochs)
cfg = Config()
cfg.BIDMC_DIR = "/kaggle/input/datasets/oshanimaduwage/bidmc-ppg-and-respiration-dataset-1-0-0/bidmc-ppg-and-respiration-dataset-1.0.0"
cfg.WINDOW_S = 3.0    
cfg.STRIDE_S = 0.30
cfg.BATCH_SIZE = 64
cfg.EPOCHS = 5
cfg.LR = 5e-4
cfg.ALPHA = 0.50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Testing on: {device}")

# 2. Patch the Architecture (Using clean Subclassing instead of Monkey-patching)
class RobustRegressionHead(nn.Module):
    def __init__(self, in_features, num_transitions, window_samples, dropout):
        super().__init__()
        self.window_samples = float(window_samples)
        self.net = nn.Sequential(
            nn.Linear(in_features, in_features // 2),
            nn.LayerNorm(in_features // 2), 
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(in_features // 2, num_transitions),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x) * self.window_samples

# Create a clean subclass that overrides the parent's heads safely
class RobustRespiratoryDetector(RespiratoryDetector):
    def __init__(self, cfg):
        # Build the original architecture first
        super().__init__(cfg)
        # Safely overwrite the specific regression heads
        self.reg_fusion = RobustRegressionHead(cfg.DENSE_UNITS, cfg.NUM_TRANSITIONS, cfg.WINDOW_SAMPLES, cfg.DROPOUT)
        self.reg_primary = RobustRegressionHead(cfg.DENSE_UNITS, cfg.NUM_TRANSITIONS, cfg.WINDOW_SAMPLES, cfg.DROPOUT)

# 3. The Modified Loss Function (The 10x Multiplier)
class MaskedCompositeLoss(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.alpha = cfg.ALPHA
        self.window_samples = float(cfg.WINDOW_SAMPLES)
        self.register_buffer("pos_weight", torch.full((cfg.NUM_TRANSITIONS,), cfg.POS_WEIGHT))

    def forward(self, preds, targets):
        t_prob, t_loc = targets["t_prob"].float(), targets["t_loc"].float()
        
        # Classification
        bce = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(t_prob.device))
        l_cls = (bce(preds["logits_fusion"], t_prob) + bce(preds["logits_primary"], t_prob)) / 2.0

        # Regression
        mask = (t_loc >= 0.0).float()
        loc_gt_safe = t_loc.clamp(min=0.0)
        n_valid = mask.sum().clamp(min=1.0)

        loc_fusion_norm = preds["loc"] / self.window_samples
        loc_gt_norm     = loc_gt_safe / self.window_samples
        l1_fusion = nn.functional.smooth_l1_loss(loc_fusion_norm, loc_gt_norm, reduction='none')
        l_reg_fusion = (l1_fusion * mask).sum() / n_valid

        if preds["loc_primary"] is not None:
            loc_primary_norm = preds["loc_primary"] / self.window_samples
            l1_primary = nn.functional.smooth_l1_loss(loc_primary_norm, loc_gt_norm, reduction='none')
            l_reg_primary = (l1_primary * mask).sum() / n_valid
            
            # ---> THE CRITICAL FIX: FORCE MULTIPLIER HERE <---
            l_reg = ((l_reg_fusion + l_reg_primary) / 2.0) * 10.0
        else:
            # ---> THE CRITICAL FIX: FORCE MULTIPLIER HERE <---
            l_reg = l_reg_fusion * 10.0

        return {"total": self.alpha * l_cls + (1.0 - self.alpha) * l_reg, "cls": l_cls, "reg": l_reg}

# 4. Load Data
print("Loading data...")
bidmc_loader = BIDMCLoader(cfg)
all_recordings = bidmc_loader.load_all()
train_recs, val_recs = subject_split(all_recordings, val_fraction=0.15, seed=42)

train_ds = RespiratoryWindowDataset(train_recs, cfg, augment=True)
val_ds   = RespiratoryWindowDataset(val_recs,   cfg, augment=False)
train_loader = make_loader(train_ds, cfg, train=True,  num_workers=2)
val_loader   = make_loader(val_ds,   cfg, train=False, num_workers=2)

# 5. Initialize Model & Training Tools (USING THE SUBCLASS)
model = RobustRespiratoryDetector(cfg).to(device)
dummy_input = torch.zeros(2, 1, cfg.WINDOW_SAMPLES, device=device)
_ = model(dummy_input, dummy_input, dummy_input)

loss_fn = MaskedCompositeLoss(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler()

# 6. The 5-Epoch Loop
print("\n--- Starting 5-Epoch Diagnostic Run ---")
for epoch in range(1, cfg.EPOCHS + 1):
    
    # Train
    model.train()
    tr_totals = defaultdict(float)
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        p, a, m = batch["primary"].to(device), batch["auxiliary"].to(device), batch["aux_marker1"].to(device)
        t_prob, t_loc = batch["t_prob"].to(device), batch["t_loc"].to(device)

        with torch.autocast('cuda', enabled=True):
            preds = model(p, a, m)
            losses = loss_fn(preds, {"t_prob": t_prob, "t_loc": t_loc})

        scaler.scale(losses["total"]).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        scaler.step(optimizer)
        scaler.update()

        for k, v in losses.items():
            tr_totals[k] += v.item()
            
    # Evaluate
    model.eval()
    val_totals, all_probs, all_labels = defaultdict(float), [], []
    total_mae_ms, valid_batches = 0.0, 0
    
    with torch.no_grad():
        for batch in val_loader:
            p, a, m = batch["primary"].to(device), batch["auxiliary"].to(device), batch["aux_marker1"].to(device)
            t_prob, t_loc = batch["t_prob"].to(device), batch["t_loc"].to(device)
            
            with torch.autocast('cuda', enabled=True):
                preds = model(p, a, m)
                losses = loss_fn(preds, {"t_prob": t_prob, "t_loc": t_loc})
            
            for k, v in losses.items():
                val_totals[k] += v.item()

            mask = (t_loc >= 0.0)
            if mask.any():
                err_samples = torch.abs(preds["loc"][mask] - t_loc[mask])
                total_mae_ms += ((err_samples / cfg.TARGET_FS) * 1000.0).mean().item()
                valid_batches += 1

            all_probs.append(preds["prob"].cpu())
            all_labels.append(t_prob.cpu())

    val_mae = total_mae_ms / valid_batches if valid_batches > 0 else 999.0
    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    preds_bin = (probs >= 0.5).astype(int)
    val_f1 = f1_score(labels[:, 0], preds_bin[:, 0], zero_division=0)
    
    tr_reg = tr_totals["reg"] / len(train_loader)
    va_reg = val_totals["reg"] / len(val_loader)
    
    print(f"Ep {epoch} | Tr_Reg: {tr_reg:.3f} | Va_Reg: {va_reg:.3f} | F1: {val_f1:.3f} | MAE: {val_mae:.1f}ms")

Testing on: cuda
Loading data...
BIDMC: loaded 53 / 53 recordings.

--- Starting 5-Epoch Diagnostic Run ---


/tmp/ipykernel_144/3906723337.py:108: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Ep 1 | Tr_Reg: 0.383 | Va_Reg: 0.392 | F1: 0.942 | MAE: 709.6ms
Ep 2 | Tr_Reg: 0.362 | Va_Reg: 0.440 | F1: 0.935 | MAE: 749.2ms
Ep 3 | Tr_Reg: 0.352 | Va_Reg: 0.444 | F1: 0.934 | MAE: 740.0ms
Ep 4 | Tr_Reg: 0.339 | Va_Reg: 0.445 | F1: 0.933 | MAE: 727.8ms
Ep 5 | Tr_Reg: 0.329 | Va_Reg: 0.448 | F1: 0.938 | MAE: 731.3ms


In [5]:
# ==============================================================================
# ARCHITECTURAL FIX: Spatial Preservation & 1D Soft-Argmax
# ==============================================================================
import torch
import torch.nn as nn
from PPG_Breath_Model_Architecture import Encoder, ConvBlock

class ConvCore(nn.Module):
    """A pure CNN core that maintains the (B, C, T) temporal layout. No flattening."""
    def __init__(self, in_ch, conv_channels):
        super().__init__()
        layers = []
        ch = in_ch
        for i, out_ch in enumerate(conv_channels):
            use_pool = (i < len(conv_channels) - 1)
            layers.append(ConvBlock(ch, out_ch, kernel=5, pool=use_pool))
            ch = out_ch
        self.convs = nn.Sequential(*layers)

    def forward(self, x):
        return self.convs(x)  # Returns (B, C, T)

class LazyClassificationHead(nn.Module):
    """Flattens the sequence and lazily builds the dense classifier."""
    def __init__(self, num_transitions, dropout):
        super().__init__()
        self.num_transitions = num_transitions
        self.dropout_p = dropout
        self.net = None

    def forward(self, x):
        x = torch.flatten(x, 1)  # Flatten (B, C, T) -> (B, C * T)
        if self.net is None:
            in_f = x.shape[1]
            self.net = nn.Sequential(
                nn.Linear(in_f, in_f // 2),
                nn.BatchNorm1d(in_f // 2),
                nn.ReLU(inplace=True),
                nn.Dropout(self.dropout_p),
                nn.Linear(in_f // 2, self.num_transitions)
            ).to(x.device)
        return self.net(x)

class TemporalRegressionHead(nn.Module):
    """Uses 1D Soft-Argmax to find the temporal 'Center of Mass' of the breath."""
    def __init__(self, in_channels, num_transitions, window_samples):
        super().__init__()
        self.window_samples = float(window_samples)
        # 1x1 Conv to squash features into heatmaps for the 2 transitions
        self.heatmap_conv = nn.Conv1d(in_channels, num_transitions, kernel_size=1)

    def forward(self, x):
        # x is (B, C, T)
        B, C, T = x.shape
        
        # 1. Generate temporal logits: (B, 2, T)
        logits = self.heatmap_conv(x)
        
        # 2. Convert to probability distribution across time
        probs = torch.softmax(logits, dim=-1)
        
        # 3. Create a temporal grid matching the current resolution
        grid = torch.linspace(0, self.window_samples, T, device=x.device)
        
        # 4. Calculate Expected Value (Center of Mass)
        expected_loc = torch.sum(probs * grid, dim=-1)  # Output: (B, 2)
        
        return expected_loc

class TemporalRespiratoryDetector(nn.Module):
    """The fully assembled, temporally-aware model."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        enc_ch = cfg.ENCODER_CH

        # 1. Encoders (Reused from your original file)
        self.primary_encoder = Encoder(enc_ch)
        self.support_encoder = Encoder(enc_ch)
        self.feature_encoder = Encoder(enc_ch)
        
        enc_out = self.primary_encoder.out_channels
        fusion_in_ch = enc_out * 3

        # 2. Cores (Now purely convolutional)
        self.fusion_core = ConvCore(fusion_in_ch, cfg.CORE_CH)
        self.primary_core = ConvCore(enc_out, cfg.CORE_CH)

        # 3. Heads
        self.cls_fusion = LazyClassificationHead(cfg.NUM_TRANSITIONS, cfg.DROPOUT)
        self.cls_primary = LazyClassificationHead(cfg.NUM_TRANSITIONS, cfg.DROPOUT)
        
        self.reg_fusion = TemporalRegressionHead(cfg.CORE_CH[-1], cfg.NUM_TRANSITIONS, cfg.WINDOW_SAMPLES)
        self.reg_primary = TemporalRegressionHead(cfg.CORE_CH[-1], cfg.NUM_TRANSITIONS, cfg.WINDOW_SAMPLES)

    def forward(self, primary, auxiliary, aux_marker1):
        # Encoders
        feat_ppg = self.primary_encoder(primary)
        feat_ecg = self.support_encoder(auxiliary)
        feat_rpeak = self.feature_encoder(aux_marker1)
        
        # Primary Path
        core_primary = self.primary_core(feat_ppg)  # (B, C, T)
        logits_primary = self.cls_primary(core_primary)
        loc_primary = self.reg_primary(core_primary) if self.training else None

        # Fusion Path
        fused = torch.cat([feat_ppg, feat_ecg, feat_rpeak], dim=1)
        core_fusion = self.fusion_core(fused)       # (B, C, T)
        logits_fusion = self.cls_fusion(core_fusion)
        loc_fusion = self.reg_fusion(core_fusion)

        # Classification output
        prob = torch.sigmoid((logits_fusion + logits_primary) / 2.0)

        return {
            "logits_fusion": logits_fusion,
            "logits_primary": logits_primary,
            "prob": prob,
            "loc": loc_fusion,
            "loc_primary": loc_primary
        }

# ---> IMPORTANT: Update your model initialization block <---
# model = TemporalRespiratoryDetector(cfg).to(device)

In [6]:
# ==============================================================================
# MANUAL TEST: 5-Epoch Run with 10x Regression Loss Multiplier (FIXED)
# ==============================================================================
import time
from collections import defaultdict
import torch
import torch.nn as nn
from sklearn.metrics import f1_score

from PPG_Breath_Model_Preprocessing import Config, BIDMCLoader, subject_split
from PPG_Breath_Model_Dataset import RespiratoryWindowDataset, make_loader
from PPG_Breath_Model_Architecture import RespiratoryDetector

# 1. Configuration (Restricted to 5 Epochs)
cfg = Config()
cfg.BIDMC_DIR = "/kaggle/input/datasets/oshanimaduwage/bidmc-ppg-and-respiration-dataset-1-0-0/bidmc-ppg-and-respiration-dataset-1.0.0"
cfg.WINDOW_S = 3.0    
cfg.STRIDE_S = 0.30
cfg.BATCH_SIZE = 64
cfg.EPOCHS = 5
cfg.LR = 5e-4
cfg.ALPHA = 0.50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Testing on: {device}")

# 2. Patch the Architecture (Using clean Subclassing instead of Monkey-patching)
class RobustRegressionHead(nn.Module):
    def __init__(self, in_features, num_transitions, window_samples, dropout):
        super().__init__()
        self.window_samples = float(window_samples)
        self.net = nn.Sequential(
            nn.Linear(in_features, in_features // 2),
            nn.LayerNorm(in_features // 2), 
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(in_features // 2, num_transitions),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x) * self.window_samples

# Create a clean subclass that overrides the parent's heads safely
class RobustRespiratoryDetector(RespiratoryDetector):
    def __init__(self, cfg):
        # Build the original architecture first
        super().__init__(cfg)
        # Safely overwrite the specific regression heads
        self.reg_fusion = RobustRegressionHead(cfg.DENSE_UNITS, cfg.NUM_TRANSITIONS, cfg.WINDOW_SAMPLES, cfg.DROPOUT)
        self.reg_primary = RobustRegressionHead(cfg.DENSE_UNITS, cfg.NUM_TRANSITIONS, cfg.WINDOW_SAMPLES, cfg.DROPOUT)

# 3. The Modified Loss Function (The 10x Multiplier)
class MaskedCompositeLoss(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.alpha = cfg.ALPHA
        self.window_samples = float(cfg.WINDOW_SAMPLES)
        self.register_buffer("pos_weight", torch.full((cfg.NUM_TRANSITIONS,), cfg.POS_WEIGHT))

    def forward(self, preds, targets):
        t_prob, t_loc = targets["t_prob"].float(), targets["t_loc"].float()
        
        # Classification
        bce = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(t_prob.device))
        l_cls = (bce(preds["logits_fusion"], t_prob) + bce(preds["logits_primary"], t_prob)) / 2.0

        # Regression
        mask = (t_loc >= 0.0).float()
        loc_gt_safe = t_loc.clamp(min=0.0)
        n_valid = mask.sum().clamp(min=1.0)

        loc_fusion_norm = preds["loc"] / self.window_samples
        loc_gt_norm     = loc_gt_safe / self.window_samples
        l1_fusion = nn.functional.smooth_l1_loss(loc_fusion_norm, loc_gt_norm, reduction='none')
        l_reg_fusion = (l1_fusion * mask).sum() / n_valid

        if preds["loc_primary"] is not None:
            loc_primary_norm = preds["loc_primary"] / self.window_samples
            l1_primary = nn.functional.smooth_l1_loss(loc_primary_norm, loc_gt_norm, reduction='none')
            l_reg_primary = (l1_primary * mask).sum() / n_valid
            
            # ---> THE CRITICAL FIX: FORCE MULTIPLIER HERE <---
            l_reg = ((l_reg_fusion + l_reg_primary) / 2.0) * 10.0
        else:
            # ---> THE CRITICAL FIX: FORCE MULTIPLIER HERE <---
            l_reg = l_reg_fusion * 10.0

        return {"total": self.alpha * l_cls + (1.0 - self.alpha) * l_reg, "cls": l_cls, "reg": l_reg}

# 4. Load Data
print("Loading data...")
bidmc_loader = BIDMCLoader(cfg)
all_recordings = bidmc_loader.load_all()
train_recs, val_recs = subject_split(all_recordings, val_fraction=0.15, seed=42)

train_ds = RespiratoryWindowDataset(train_recs, cfg, augment=True)
val_ds   = RespiratoryWindowDataset(val_recs,   cfg, augment=False)
train_loader = make_loader(train_ds, cfg, train=True,  num_workers=2)
val_loader   = make_loader(val_ds,   cfg, train=False, num_workers=2)

# 5. Initialize Model & Training Tools (USING THE SUBCLASS)
model = TemporalRespiratoryDetector(cfg).to(device)
dummy_input = torch.zeros(2, 1, cfg.WINDOW_SAMPLES, device=device)
_ = model(dummy_input, dummy_input, dummy_input)

loss_fn = MaskedCompositeLoss(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler()

# 6. The 5-Epoch Loop
print("\n--- Starting 5-Epoch Diagnostic Run ---")
for epoch in range(1, cfg.EPOCHS + 1):
    
    # Train
    model.train()
    tr_totals = defaultdict(float)
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        p, a, m = batch["primary"].to(device), batch["auxiliary"].to(device), batch["aux_marker1"].to(device)
        t_prob, t_loc = batch["t_prob"].to(device), batch["t_loc"].to(device)

        with torch.autocast('cuda', enabled=True):
            preds = model(p, a, m)
            losses = loss_fn(preds, {"t_prob": t_prob, "t_loc": t_loc})

        scaler.scale(losses["total"]).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        scaler.step(optimizer)
        scaler.update()

        for k, v in losses.items():
            tr_totals[k] += v.item()
            
    # Evaluate
    model.eval()
    val_totals, all_probs, all_labels = defaultdict(float), [], []
    total_mae_ms, valid_batches = 0.0, 0
    
    with torch.no_grad():
        for batch in val_loader:
            p, a, m = batch["primary"].to(device), batch["auxiliary"].to(device), batch["aux_marker1"].to(device)
            t_prob, t_loc = batch["t_prob"].to(device), batch["t_loc"].to(device)
            
            with torch.autocast('cuda', enabled=True):
                preds = model(p, a, m)
                losses = loss_fn(preds, {"t_prob": t_prob, "t_loc": t_loc})
            
            for k, v in losses.items():
                val_totals[k] += v.item()

            mask = (t_loc >= 0.0)
            if mask.any():
                err_samples = torch.abs(preds["loc"][mask] - t_loc[mask])
                total_mae_ms += ((err_samples / cfg.TARGET_FS) * 1000.0).mean().item()
                valid_batches += 1

            all_probs.append(preds["prob"].cpu())
            all_labels.append(t_prob.cpu())

    val_mae = total_mae_ms / valid_batches if valid_batches > 0 else 999.0
    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    preds_bin = (probs >= 0.5).astype(int)
    val_f1 = f1_score(labels[:, 0], preds_bin[:, 0], zero_division=0)
    
    tr_reg = tr_totals["reg"] / len(train_loader)
    va_reg = val_totals["reg"] / len(val_loader)
    
    print(f"Ep {epoch} | Tr_Reg: {tr_reg:.3f} | Va_Reg: {va_reg:.3f} | F1: {val_f1:.3f} | MAE: {val_mae:.1f}ms")

Testing on: cuda
Loading data...
BIDMC: loaded 53 / 53 recordings.

--- Starting 5-Epoch Diagnostic Run ---


/tmp/ipykernel_144/417832221.py:108: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Ep 1 | Tr_Reg: 0.372 | Va_Reg: 0.406 | F1: 0.926 | MAE: 714.8ms
Ep 2 | Tr_Reg: 0.351 | Va_Reg: 0.418 | F1: 0.934 | MAE: 719.5ms
Ep 3 | Tr_Reg: 0.337 | Va_Reg: 0.432 | F1: 0.935 | MAE: 725.2ms
Ep 4 | Tr_Reg: 0.329 | Va_Reg: 0.441 | F1: 0.938 | MAE: 735.5ms
Ep 5 | Tr_Reg: 0.321 | Va_Reg: 0.430 | F1: 0.939 | MAE: 719.9ms


In [7]:
# ==============================================================================
# DIAGNOSTIC TOOL: Inspecting Data Integrity & Target Scale
# ==============================================================================
import torch

print("=" * 60)
print("🎯 BATCH DIAGNOSTICS: TARGETS vs PREDICTIONS")
print("=" * 60)
print(f"Expected Window Size (cfg.WINDOW_SAMPLES): {cfg.WINDOW_SAMPLES}\n")

# 1. Put model in eval mode so we don't accidentally calculate gradients
model.eval()

# 2. Grab exactly one batch of data
batch = next(iter(train_loader))

p = batch["primary"].to(device)
a = batch["auxiliary"].to(device)
m = batch["aux_marker1"].to(device)
t_loc_target = batch["t_loc"].to(device)

# 3. Run the forward pass
with torch.no_grad():
    with torch.autocast('cuda', enabled=(device.type == 'cuda')):
        preds = model(p, a, m)

predicted_loc = preds["loc"]

# 4. Print the raw math for the first 10 windows in the batch
for i in range(min(10, cfg.BATCH_SIZE)):
    # Ground Truth Targets
    gt_tr1 = t_loc_target[i, 0].item()
    gt_tr2 = t_loc_target[i, 1].item()
    
    # Model Predictions
    pred_tr1 = predicted_loc[i, 0].item()
    pred_tr2 = predicted_loc[i, 1].item()
    
    # Formatting to highlight valid transitions vs sentinel (-1) values
    gt_str = f"TR1: {gt_tr1:>8.2f} | TR2: {gt_tr2:>8.2f}"
    pr_str = f"TR1: {pred_tr1:>8.2f} | TR2: {pred_tr2:>8.2f}"
    
    print(f"Window {i+1:02d} | Target -> {gt_str}  ||  Predict -> {pr_str}")

# 5. Global Batch Checks
max_gt = t_loc_target.max().item()
print("\n" + "-" * 60)
print(f"Maximum target value in this batch: {max_gt:.2f}")

if 0.0 < max_gt <= 1.0:
    print("🚨 FATAL ERROR: Double Normalization Detected!")
    print("   Your ground truth targets are scaled between 0.0 and 1.0.")
    print(f"   The loss function is trying to compare them against predictions scaled up to {cfg.WINDOW_SAMPLES}.")
elif max_gt == -1.0:
    print("🚨 FATAL ERROR: Sentinel Bleed!")
    print("   Every single window in this batch is empty (-1.0). Your WeightedRandomSampler is failing to supply positive examples.")
else:
    print("✅ Target scaling looks mathematically correct (values > 1.0).")
print("=" * 60)

🎯 BATCH DIAGNOSTICS: TARGETS vs PREDICTIONS
Expected Window Size (cfg.WINDOW_SAMPLES): 300

Window 01 | Target -> TR1:   276.00 | TR2:   163.00  ||  Predict -> TR1:   138.79 | TR2:   178.74
Window 02 | Target -> TR1:    -1.00 | TR2:    -1.00  ||  Predict -> TR1:    97.94 | TR2:   184.10
Window 03 | Target -> TR1:   263.00 | TR2:    -1.00  ||  Predict -> TR1:   160.00 | TR2:   140.81
Window 04 | Target -> TR1:    -1.00 | TR2:    -1.00  ||  Predict -> TR1:   126.57 | TR2:   176.36
Window 05 | Target -> TR1:    -1.00 | TR2:    -1.00  ||  Predict -> TR1:   132.39 | TR2:   151.18
Window 06 | Target -> TR1:   250.00 | TR2:   105.00  ||  Predict -> TR1:   101.27 | TR2:   159.98
Window 07 | Target -> TR1:    -1.00 | TR2:    -1.00  ||  Predict -> TR1:   131.51 | TR2:   185.73
Window 08 | Target -> TR1:    23.00 | TR2:   225.00  ||  Predict -> TR1:    91.70 | TR2:   179.57
Window 09 | Target -> TR1:    -1.00 | TR2:    -1.00  ||  Predict -> TR1:   205.53 | TR2:   122.20
Window 10 | Target -> TR1:

In [9]:
# ==============================================================================
# ARCHITECTURAL FIX: 1D Soft-Argmax (Temporal Center of Mass)
# ==============================================================================
import time
from collections import defaultdict
import torch
import torch.nn as nn
from sklearn.metrics import f1_score

from PPG_Breath_Model_Preprocessing import Config, BIDMCLoader, subject_split
from PPG_Breath_Model_Dataset import RespiratoryWindowDataset, make_loader
from PPG_Breath_Model_Architecture import Encoder, ConvBlock

# 1. Configuration
cfg = Config()
cfg.BIDMC_DIR = "/kaggle/input/datasets/oshanimaduwage/bidmc-ppg-and-respiration-dataset-1-0-0/bidmc-ppg-and-respiration-dataset-1.0.0"
cfg.WINDOW_S = 3.0    
cfg.STRIDE_S = 0.30
cfg.BATCH_SIZE = 64
cfg.EPOCHS = 5
cfg.LR = 5e-4
cfg.ALPHA = 0.50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Testing on: {device}")

# 2. The New Temporal Architecture Components
class ConvCore(nn.Module):
    """Maintains the (B, C, T) temporal layout. No flattening."""
    def __init__(self, in_ch, conv_channels):
        super().__init__()
        layers = []
        ch = in_ch
        for i, out_ch in enumerate(conv_channels):
            use_pool = (i < len(conv_channels) - 1)
            layers.append(ConvBlock(ch, out_ch, kernel=5, pool=use_pool))
            ch = out_ch
        self.convs = nn.Sequential(*layers)

    def forward(self, x):
        return self.convs(x)

class LazyClassificationHead(nn.Module):
    """Flattens the sequence and lazily builds the dense classifier."""
    def __init__(self, num_transitions, dropout):
        super().__init__()
        self.num_transitions = num_transitions
        self.dropout_p = dropout
        self.net = None

    def forward(self, x):
        x = torch.flatten(x, 1)  # Flatten (B, C, T) -> (B, C * T)
        if self.net is None:
            in_f = x.shape[1]
            self.net = nn.Sequential(
                nn.Linear(in_f, in_f // 2),
                nn.BatchNorm1d(in_f // 2),
                nn.ReLU(inplace=True),
                nn.Dropout(self.dropout_p),
                nn.Linear(in_f // 2, self.num_transitions)
            ).to(x.device)
        return self.net(x)

class TemporalRegressionHead(nn.Module):
    """Uses 1D Soft-Argmax to find the temporal 'Center of Mass'."""
    def __init__(self, in_channels, num_transitions, window_samples):
        super().__init__()
        self.window_samples = float(window_samples)
        # 1x1 Conv squashes features into 2 heatmaps (insp/exp)
        self.heatmap_conv = nn.Conv1d(in_channels, num_transitions, kernel_size=1)

    def forward(self, x):
        B, C, T = x.shape
        logits = self.heatmap_conv(x)
        probs = torch.softmax(logits, dim=-1)
        # Map the T resolution bins back to the original 300-sample grid
        grid = torch.linspace(0, self.window_samples - 1, T, device=x.device)
        expected_loc = torch.sum(probs * grid, dim=-1)  
        return expected_loc

class TemporalRespiratoryDetector(nn.Module):
    """Fully assembled, temporally-aware model."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        enc_ch = cfg.ENCODER_CH

        self.primary_encoder = Encoder(enc_ch)
        self.support_encoder = Encoder(enc_ch)
        self.feature_encoder = Encoder(enc_ch)
        
        enc_out = self.primary_encoder.out_channels
        fusion_in_ch = enc_out * 3

        self.fusion_core = ConvCore(fusion_in_ch, cfg.CORE_CH)
        self.primary_core = ConvCore(enc_out, cfg.CORE_CH)

        self.cls_fusion = LazyClassificationHead(cfg.NUM_TRANSITIONS, cfg.DROPOUT)
        self.cls_primary = LazyClassificationHead(cfg.NUM_TRANSITIONS, cfg.DROPOUT)
        
        self.reg_fusion = TemporalRegressionHead(cfg.CORE_CH[-1], cfg.NUM_TRANSITIONS, cfg.WINDOW_SAMPLES)
        self.reg_primary = TemporalRegressionHead(cfg.CORE_CH[-1], cfg.NUM_TRANSITIONS, cfg.WINDOW_SAMPLES)

    def forward(self, primary, auxiliary, aux_marker1):
        feat_ppg = self.primary_encoder(primary)
        feat_ecg = self.support_encoder(auxiliary)
        feat_rpeak = self.feature_encoder(aux_marker1)
        
        core_primary = self.primary_core(feat_ppg) 
        logits_primary = self.cls_primary(core_primary)
        loc_primary = self.reg_primary(core_primary) if self.training else None

        fused = torch.cat([feat_ppg, feat_ecg, feat_rpeak], dim=1)
        core_fusion = self.fusion_core(fused)
        logits_fusion = self.cls_fusion(core_fusion)
        loc_fusion = self.reg_fusion(core_fusion)

        prob = torch.sigmoid((logits_fusion + logits_primary) / 2.0)

        return {
            "logits_fusion": logits_fusion,
            "logits_primary": logits_primary,
            "prob": prob,
            "loc": loc_fusion,
            "loc_primary": loc_primary
        }

# 3. Masked Loss with Heavy Regression Priority
class MaskedCompositeLoss(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.alpha = cfg.ALPHA
        self.window_samples = float(cfg.WINDOW_SAMPLES)
        self.register_buffer("pos_weight", torch.full((cfg.NUM_TRANSITIONS,), cfg.POS_WEIGHT))

    def forward(self, preds, targets):
        t_prob, t_loc = targets["t_prob"].float(), targets["t_loc"].float()
        
        bce = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(t_prob.device))
        l_cls = (bce(preds["logits_fusion"], t_prob) + bce(preds["logits_primary"], t_prob)) / 2.0

        mask = (t_loc >= 0.0).float()
        loc_gt_safe = t_loc.clamp(min=0.0)
        n_valid = mask.sum().clamp(min=1.0)

        loc_fusion_norm = preds["loc"] / self.window_samples
        loc_gt_norm     = loc_gt_safe / self.window_samples
        l1_fusion = nn.functional.smooth_l1_loss(loc_fusion_norm, loc_gt_norm, reduction='none')
        l_reg_fusion = (l1_fusion * mask).sum() / n_valid

        if preds["loc_primary"] is not None:
            loc_primary_norm = preds["loc_primary"] / self.window_samples
            l1_primary = nn.functional.smooth_l1_loss(loc_primary_norm, loc_gt_norm, reduction='none')
            l_reg_primary = (l1_primary * mask).sum() / n_valid
            l_reg = ((l_reg_fusion + l_reg_primary) / 2.0) * 10.0
        else:
            l_reg = l_reg_fusion * 10.0

        return {"total": self.alpha * l_cls + (1.0 - self.alpha) * l_reg, "cls": l_cls, "reg": l_reg}

# 4. Execute Pipeline
print("Loading data...")
bidmc_loader = BIDMCLoader(cfg)
all_recordings = bidmc_loader.load_all()
train_recs, val_recs = subject_split(all_recordings, val_fraction=0.15, seed=42)

train_ds = RespiratoryWindowDataset(train_recs, cfg, augment=True)
val_ds   = RespiratoryWindowDataset(val_recs,   cfg, augment=False)
train_loader = make_loader(train_ds, cfg, train=True,  num_workers=2)
val_loader   = make_loader(val_ds,   cfg, train=False, num_workers=2)

model = TemporalRespiratoryDetector(cfg).to(device)
dummy_input = torch.zeros(2, 1, cfg.WINDOW_SAMPLES, device=device)
_ = model(dummy_input, dummy_input, dummy_input)

loss_fn = MaskedCompositeLoss(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler()

print("\n--- Starting 5-Epoch Temporal Diagnostic Run ---")
for epoch in range(1, cfg.EPOCHS + 1):
    model.train()
    tr_totals = defaultdict(float)
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        p, a, m = batch["primary"].to(device), batch["auxiliary"].to(device), batch["aux_marker1"].to(device)
        t_prob, t_loc = batch["t_prob"].to(device), batch["t_loc"].to(device)

        with torch.autocast('cuda', enabled=True):
            preds = model(p, a, m)
            losses = loss_fn(preds, {"t_prob": t_prob, "t_loc": t_loc})

        scaler.scale(losses["total"]).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        scaler.step(optimizer)
        scaler.update()

        for k, v in losses.items():
            tr_totals[k] += v.item()
            
    model.eval()
    val_totals, all_probs, all_labels = defaultdict(float), [], []
    total_mae_ms, valid_batches = 0.0, 0
    
    with torch.no_grad():
        for batch in val_loader:
            p, a, m = batch["primary"].to(device), batch["auxiliary"].to(device), batch["aux_marker1"].to(device)
            t_prob, t_loc = batch["t_prob"].to(device), batch["t_loc"].to(device)
            
            with torch.autocast('cuda', enabled=True):
                preds = model(p, a, m)
                losses = loss_fn(preds, {"t_prob": t_prob, "t_loc": t_loc})
            
            for k, v in losses.items():
                val_totals[k] += v.item()

            mask = (t_loc >= 0.0)
            if mask.any():
                err_samples = torch.abs(preds["loc"][mask] - t_loc[mask])
                total_mae_ms += ((err_samples / cfg.TARGET_FS) * 1000.0).mean().item()
                valid_batches += 1

            all_probs.append(preds["prob"].cpu())
            all_labels.append(t_prob.cpu())

    val_mae = total_mae_ms / valid_batches if valid_batches > 0 else 999.0
    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    preds_bin = (probs >= 0.5).astype(int)
    val_f1 = f1_score(labels[:, 0], preds_bin[:, 0], zero_division=0)
    
    tr_reg = tr_totals["reg"] / len(train_loader)
    va_reg = val_totals["reg"] / len(val_loader)
    
    print(f"Ep {epoch} | Tr_Reg: {tr_reg:.3f} | Va_Reg: {va_reg:.3f} | F1: {val_f1:.3f} | MAE: {val_mae:.1f}ms")

Testing on: cuda
Loading data...
BIDMC: loaded 53 / 53 recordings.

--- Starting 5-Epoch Temporal Diagnostic Run ---


/tmp/ipykernel_144/73307212.py:178: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Ep 1 | Tr_Reg: 0.369 | Va_Reg: 0.407 | F1: 0.936 | MAE: 717.6ms
Ep 2 | Tr_Reg: 0.349 | Va_Reg: 0.442 | F1: 0.936 | MAE: 737.4ms
Ep 3 | Tr_Reg: 0.338 | Va_Reg: 0.444 | F1: 0.937 | MAE: 740.7ms
Ep 4 | Tr_Reg: 0.328 | Va_Reg: 0.455 | F1: 0.937 | MAE: 732.9ms
Ep 5 | Tr_Reg: 0.321 | Va_Reg: 0.453 | F1: 0.937 | MAE: 742.8ms
